# NovaSuite — Data Cleaning Notebook
**Project:** B2B SaaS Pricing Optimization & Revenue Growth  
**Datasets:** customers, plans, revenue, usage, churn  
**Libraries:** pandas, numpy only  
**Date Range:** January 2021 – June 2024

---
### Notebook Structure
1. Imports & Load Data
2. Schema Inspection
3. Missing Value Audit
4. Duplicate Detection & Removal
5. Data Type Corrections
6. Domain / Business Rule Validation
7. Outlier Detection
8. Referential Integrity Checks (cross-table)
9. Feature Engineering (derived cleaning columns)
10. Save Cleaned Datasets
11. Cleaning Summary Report

---
## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np

# ── Load all five datasets ──────────────────────────────────────────────────
customers = pd.read_csv("customers.csv")
plans     = pd.read_csv("plans.csv")
revenue   = pd.read_csv("revenue.csv")
usage     = pd.read_csv("usage.csv")
churn     = pd.read_csv("churn.csv")

print("Loaded shapes:")
for name, df in [("customers", customers), ("plans", plans),
                  ("revenue", revenue), ("usage", usage), ("churn", churn)]:
    print(f"  {name:12s}: {df.shape[0]:>6,} rows  ×  {df.shape[1]} cols")

---
## 2. Schema Inspection
Quick look at column names, dtypes, and sample rows for every table.

In [ ]:
# ── Helper: compact schema view ─────────────────────────────────────────────
def schema_info(df, name):
    print(f"\n{'='*60}")
    print(f" TABLE: {name}")
    print(f"{'='*60}")
    info = pd.DataFrame({
        "dtype"      : df.dtypes,
        "non_null"   : df.notna().sum(),
        "null_count" : df.isna().sum(),
        "null_%"     : (df.isna().mean() * 100).round(2),
        "unique"     : df.nunique(),
        "sample"     : df.iloc[0]
    })
    print(info.to_string())

for name, df in [("customers", customers), ("plans", plans),
                  ("revenue", revenue), ("usage", usage), ("churn", churn)]:
    schema_info(df, name)

In [ ]:
# ── Preview each table ───────────────────────────────────────────────────────
print("\n--- customers (head 3) ---"); print(customers.head(3).to_string())
print("\n--- plans ---");             print(plans.to_string())
print("\n--- revenue (head 3) ---");  print(revenue.head(3).to_string())
print("\n--- usage (head 3) ---");    print(usage.head(3).to_string())
print("\n--- churn ---");             print(churn.to_string())

---
## 3. Missing Value Audit

In [ ]:
# ── 3.1  Consolidated null heatmap across all tables ────────────────────────
def null_summary(df, name):
    s = df.isna().sum()
    s = s[s > 0]
    if s.empty:
        print(f"[{name}]  ✓  No missing values")
    else:
        pct = (s / len(df) * 100).round(2)
        print(f"\n[{name}]  ⚠  Missing values found:")
        for col in s.index:
            print(f"   {col:30s}: {s[col]:>5} nulls  ({pct[col]:.2f}%)")

for name, df in [("customers", customers), ("plans", plans),
                  ("revenue", revenue), ("usage", usage), ("churn", churn)]:
    null_summary(df, name)

In [ ]:
# ── 3.2  Check for whitespace-only or empty-string cells (hidden nulls) ─────
def hidden_nulls(df, name):
    str_cols = df.select_dtypes(include="object").columns
    found = {}
    for col in str_cols:
        mask = df[col].astype(str).str.strip() == ""
        if mask.sum() > 0:
            found[col] = mask.sum()
    if found:
        print(f"[{name}]  hidden empty strings: {found}")
    else:
        print(f"[{name}]  ✓  No hidden empty strings")

for name, df in [("customers", customers), ("plans", plans),
                  ("revenue", revenue), ("usage", usage), ("churn", churn)]:
    hidden_nulls(df, name)

In [ ]:
# ── 3.3  Replace any empty strings with NaN (safe no-op if none exist) ──────
for df in [customers, plans, revenue, usage, churn]:
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(
        lambda c: c.str.strip().replace("", np.nan)
    )

print("Empty strings replaced with NaN (if any existed).")

---
## 4. Duplicate Detection & Removal

In [ ]:
# ── 4.1  Full-row duplicates ─────────────────────────────────────────────────
datasets = {"customers": customers, "plans": plans,
            "revenue": revenue, "usage": usage, "churn": churn}

for name, df in datasets.items():
    n_dup = df.duplicated().sum()
    print(f"[{name}]  full-row duplicates: {n_dup}")

In [ ]:
# ── 4.2  Key-level duplicates ────────────────────────────────────────────────
# customers   → customer_id must be unique (1 row per customer)
# plans       → plan_id must be unique
# revenue     → (customer_id, month) must be unique
# usage       → (customer_id, month) must be unique
# churn       → customer_id must be unique (can only churn once)

key_checks = [
    ("customers", customers, ["customer_id"]),
    ("plans",     plans,     ["plan_id"]),
    ("revenue",   revenue,   ["customer_id", "month"]),
    ("usage",     usage,     ["customer_id", "month"]),
    ("churn",     churn,     ["customer_id"]),
]

for name, df, keys in key_checks:
    n_dup = df.duplicated(subset=keys).sum()
    status = "✓" if n_dup == 0 else "⚠"
    print(f"[{name}]  {status}  key duplicates on {keys}: {n_dup}")

In [ ]:
# ── 4.3  Drop full-row duplicates (keep first occurrence) ────────────────────
before = {name: len(df) for name, df in datasets.items()}

customers = customers.drop_duplicates().reset_index(drop=True)
plans     = plans.drop_duplicates().reset_index(drop=True)
revenue   = revenue.drop_duplicates().reset_index(drop=True)
usage     = usage.drop_duplicates().reset_index(drop=True)
churn     = churn.drop_duplicates().reset_index(drop=True)

after = {"customers": len(customers), "plans": len(plans),
         "revenue": len(revenue), "usage": len(usage), "churn": len(churn)}

print("Rows removed by deduplication:")
for name in before:
    print(f"  {name:12s}: {before[name] - after[name]} rows removed  "
          f"({after[name]:,} remaining)")

---
## 5. Data Type Corrections

In [ ]:
# ── 5.1  Date columns → datetime ─────────────────────────────────────────────
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
revenue["month"]         = pd.to_datetime(revenue["month"],         errors="coerce")
usage["month"]           = pd.to_datetime(usage["month"],           errors="coerce")
churn["churn_date"]      = pd.to_datetime(churn["churn_date"],      errors="coerce")

# Report any dates that failed parsing (coerced to NaT)
date_cols = [
    ("customers", customers, "signup_date"),
    ("revenue",   revenue,   "month"),
    ("usage",     usage,     "month"),
    ("churn",     churn,     "churn_date"),
]
for tbl, df, col in date_cols:
    n_nat = df[col].isna().sum()
    print(f"[{tbl}]  {col}: {n_nat} NaT after parse "
          f"{'(⚠ investigate!)' if n_nat else '✓'}")

In [ ]:
# ── 5.2  Categorical / string columns → clean, consistent case ───────────────
str_strip = {
    "customers" : ["industry", "company_size", "country", "plan_id"],
    "plans"     : ["plan_id", "plan_name"],
    "revenue"   : ["plan_id", "renewal_status"],
    "usage"     : ["top_feature"],
    "churn"     : ["reason"],
}

df_map = {"customers": customers, "plans": plans,
          "revenue": revenue, "usage": usage, "churn": churn}

for tbl, cols in str_strip.items():
    for col in cols:
        df_map[tbl][col] = df_map[tbl][col].astype(str).str.strip()

print("String columns stripped of leading/trailing whitespace.")

In [ ]:
# ── 5.3  Numeric columns – coerce & verify ────────────────────────────────────
num_cols = {
    "plans"   : ["monthly_price", "max_users"],
    "revenue" : ["base_revenue", "discount", "expansion_revenue",
                 "subscription_revenue", "tenure_months"],
    "usage"   : ["active_users", "sessions", "login_count", "feature_usage"],
}

for tbl, cols in num_cols.items():
    for col in cols:
        original_nulls = df_map[tbl][col].isna().sum()
        df_map[tbl][col] = pd.to_numeric(df_map[tbl][col], errors="coerce")
        new_nulls = df_map[tbl][col].isna().sum()
        if new_nulls > original_nulls:
            print(f"[{tbl}]  ⚠  {col}: {new_nulls - original_nulls} "
                  "values could not be coerced to numeric")
        else:
            print(f"[{tbl}]  ✓  {col}")

---
## 6. Domain / Business Rule Validation

In [ ]:
# ── 6.1  customers — allowed categorical values ───────────────────────────────
VALID_INDUSTRIES  = {"FinTech", "Healthcare", "E-Commerce", "EdTech",
                     "Logistics", "SaaS/Tech", "Retail", "Manufacturing"}
VALID_SIZES       = {"Small", "Medium", "Large", "Enterprise"}
VALID_COUNTRIES   = {"USA", "India", "UK", "Germany",
                     "Canada", "Australia", "Singapore", "UAE"}
VALID_PLAN_IDS    = set(plans["plan_id"].unique())

checks = [
    ("industry",     VALID_INDUSTRIES),
    ("company_size", VALID_SIZES),
    ("country",      VALID_COUNTRIES),
    ("plan_id",      VALID_PLAN_IDS),
]

print("[customers] Categorical domain checks:")
for col, valid_set in checks:
    bad = customers[~customers[col].isin(valid_set)]
    if bad.empty:
        print(f"  ✓  {col}")
    else:
        print(f"  ⚠  {col}: {len(bad)} invalid values → "
              f"{bad[col].unique().tolist()}")

In [ ]:
# ── 6.2  signup_date range: Jan 2021 – Jun 2024 ───────────────────────────────
DATE_MIN = pd.Timestamp("2021-01-01")
DATE_MAX = pd.Timestamp("2024-06-30")

out_of_range = customers[
    (customers["signup_date"] < DATE_MIN) |
    (customers["signup_date"] > DATE_MAX)
]
print(f"[customers]  signup_date out of range [2021-01-01, 2024-06-30]: "
      f"{len(out_of_range)} rows")
if not out_of_range.empty:
    print(out_of_range[["customer_id", "signup_date"]].to_string())

In [ ]:
# ── 6.3  revenue — financial sanity checks ───────────────────────────────────
print("[revenue] Financial rule checks:")

# base_revenue must be positive
neg_base = revenue[revenue["base_revenue"] <= 0]
print(f"  base_revenue <= 0         : {len(neg_base)} rows")

# discount must be between 0 and base_revenue
neg_disc = revenue[revenue["discount"] < 0]
print(f"  discount < 0              : {len(neg_disc)} rows")

over_disc = revenue[revenue["discount"] > revenue["base_revenue"]]
print(f"  discount > base_revenue   : {len(over_disc)} rows")

# expansion_revenue must be >= 0
neg_exp = revenue[revenue["expansion_revenue"] < 0]
print(f"  expansion_revenue < 0     : {len(neg_exp)} rows")

# subscription_revenue = base - discount (within tolerance)
expected_sub = revenue["base_revenue"] - revenue["discount"]
mismatch = revenue[
    np.abs(revenue["subscription_revenue"] - expected_sub) > 0.02
]
print(f"  subscription_revenue mismatch (>$0.02 off): {len(mismatch)} rows")

# tenure_months must be >= 0
neg_ten = revenue[revenue["tenure_months"] < 0]
print(f"  tenure_months < 0         : {len(neg_ten)} rows")

In [ ]:
# ── 6.4  revenue — renewal_status allowed values ─────────────────────────────
VALID_STATUS = {"Active", "Churned", "At Risk", "Renewed"}
bad_status = revenue[~revenue["renewal_status"].isin(VALID_STATUS)]
print(f"[revenue]  invalid renewal_status: {len(bad_status)} rows")
if not bad_status.empty:
    print(bad_status["renewal_status"].value_counts())

In [ ]:
# ── 6.5  usage — value range checks ──────────────────────────────────────────
print("[usage] Range checks:")

neg_users = usage[usage["active_users"] < 0]
print(f"  active_users < 0      : {len(neg_users)} rows")

neg_sessions = usage[usage["sessions"] < 0]
print(f"  sessions < 0          : {len(neg_sessions)} rows")

neg_logins = usage[usage["login_count"] < 0]
print(f"  login_count < 0       : {len(neg_logins)} rows")

# feature_usage scored 0–10
bad_fu = usage[(usage["feature_usage"] < 0) | (usage["feature_usage"] > 10)]
print(f"  feature_usage ∉ [0,10]: {len(bad_fu)} rows")

In [ ]:
# ── 6.6  churn — reason allowed values ───────────────────────────────────────
VALID_REASONS = {"Budget Cut", "Lack of Features", "Poor Support",
                 "Switched to Competitor", "Too Expensive", "Other"}
bad_reasons = churn[~churn["reason"].isin(VALID_REASONS)]
print(f"[churn]  invalid reason: {len(bad_reasons)} rows")
if not bad_reasons.empty:
    print(bad_reasons[["customer_id", "reason"]].to_string())

print("\n[churn] Reason distribution:")
print(churn["reason"].value_counts())

In [ ]:
# ── 6.7  churn_date must be after signup_date ─────────────────────────────────
merged_churn = churn.merge(
    customers[["customer_id", "signup_date"]], on="customer_id", how="left"
)
bad_churn_date = merged_churn[
    merged_churn["churn_date"] < merged_churn["signup_date"]
]
print(f"[churn]  churn_date < signup_date: {len(bad_churn_date)} rows")
if not bad_churn_date.empty:
    print(bad_churn_date[["customer_id", "signup_date", "churn_date"]].to_string())

In [ ]:
# ── 6.8  plans — price & seat sanity ──────────────────────────────────────────
print("[plans] Sanity check:")
print(plans[["plan_name", "monthly_price", "max_users"]].to_string())

neg_price = plans[plans["monthly_price"] <= 0]
print(f"\n  monthly_price <= 0 : {len(neg_price)} rows")

neg_seats = plans[plans["max_users"] <= 0]
print(f"  max_users <= 0     : {len(neg_seats)} rows")

---
## 7. Outlier Detection

In [ ]:
# ── 7.1  Descriptive stats for numeric columns ────────────────────────────────
print("=== revenue numeric stats ===")
print(revenue[["base_revenue", "discount", "expansion_revenue",
               "subscription_revenue", "tenure_months"]].describe().round(2))

print("\n=== usage numeric stats ===")
print(usage[["active_users", "sessions",
             "login_count", "feature_usage"]].describe().round(2))

In [ ]:
# ── 7.2  IQR-based outlier flagging (flag, not drop) ─────────────────────────
def flag_outliers_iqr(df, cols, name):
    """Add boolean flag columns for IQR outliers (Q1-1.5*IQR, Q3+1.5*IQR)."""
    flag_counts = {}
    for col in cols:
        q1  = df[col].quantile(0.25)
        q3  = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        flag_col = f"{col}_outlier"
        df[flag_col] = (df[col] < lower) | (df[col] > upper)
        count = df[flag_col].sum()
        flag_counts[col] = count
        print(f"  [{name}]  {col:30s}: {count:>5} outliers  "
              f"(bounds [{lower:.2f}, {upper:.2f}])")
    return df

print("IQR Outlier Detection:")
revenue = flag_outliers_iqr(
    revenue,
    ["base_revenue", "discount", "expansion_revenue",
     "subscription_revenue", "tenure_months"],
    "revenue"
)
usage = flag_outliers_iqr(
    usage,
    ["active_users", "sessions", "login_count", "feature_usage"],
    "usage"
)

In [ ]:
# ── 7.3  Zero-usage months (potential at-risk signal) ─────────────────────────
zero_usage = usage[
    (usage["sessions"] == 0) & (usage["active_users"] == 0)
]
print(f"[usage]  rows with 0 sessions AND 0 active_users (dead months): "
      f"{len(zero_usage)}")
if not zero_usage.empty:
    print(zero_usage[["customer_id", "month",
                       "active_users", "sessions"]].head(10).to_string())

---
## 8. Referential Integrity Checks (Cross-Table)

In [ ]:
# ── 8.1  All customer_ids in revenue exist in customers ──────────────────────
valid_cids = set(customers["customer_id"])

orphan_revenue = revenue[~revenue["customer_id"].isin(valid_cids)]
print(f"[revenue]  customer_ids not in customers: "
      f"{orphan_revenue['customer_id'].nunique()} unique IDs, "
      f"{len(orphan_revenue)} rows")

orphan_usage = usage[~usage["customer_id"].isin(valid_cids)]
print(f"[usage]    customer_ids not in customers: "
      f"{orphan_usage['customer_id'].nunique()} unique IDs, "
      f"{len(orphan_usage)} rows")

orphan_churn = churn[~churn["customer_id"].isin(valid_cids)]
print(f"[churn]    customer_ids not in customers: "
      f"{orphan_churn['customer_id'].nunique()} unique IDs, "
      f"{len(orphan_churn)} rows")

In [ ]:
# ── 8.2  All plan_ids in customers / revenue exist in plans ──────────────────
valid_pids = set(plans["plan_id"])

bad_plan_cust = customers[~customers["plan_id"].isin(valid_pids)]
print(f"[customers] plan_id not in plans: {len(bad_plan_cust)} rows")

bad_plan_rev = revenue[~revenue["plan_id"].isin(valid_pids)]
print(f"[revenue]   plan_id not in plans: {len(bad_plan_rev)} rows")

In [ ]:
# ── 8.3  Coverage: every customer should have ≥1 revenue & usage row ─────────
cids_in_revenue = set(revenue["customer_id"].unique())
cids_in_usage   = set(usage["customer_id"].unique())

missing_rev = valid_cids - cids_in_revenue
missing_usg = valid_cids - cids_in_usage

print(f"[customers] with NO revenue rows : {len(missing_rev)}")
if missing_rev:
    print(f"  IDs: {sorted(missing_rev)[:10]} ...")

print(f"[customers] with NO usage rows   : {len(missing_usg)}")
if missing_usg:
    print(f"  IDs: {sorted(missing_usg)[:10]} ...")

In [ ]:
# ── 8.4  Revenue and usage rows should have the same (customer, month) keys ──
rev_keys   = set(zip(revenue["customer_id"], revenue["month"]))
usage_keys = set(zip(usage["customer_id"], usage["month"]))

in_rev_not_usage  = rev_keys - usage_keys
in_usage_not_rev  = usage_keys - rev_keys

print(f"(customer, month) in revenue but NOT usage : {len(in_rev_not_usage)}")
print(f"(customer, month) in usage  but NOT revenue: {len(in_usage_not_rev)}")

In [ ]:
# ── 8.5  Churned customers should have 'Churned' renewal_status ──────────────
churned_ids = set(churn["customer_id"])
churned_rev = revenue[revenue["customer_id"].isin(churned_ids)]

# For each churned customer, check if any row is marked Churned in revenue
has_churned_status = (
    churned_rev[churned_rev["renewal_status"] == "Churned"]["customer_id"]
    .unique()
)
never_marked = churned_ids - set(has_churned_status)
print(f"[integrity]  Churned customers never marked 'Churned' in revenue: "
      f"{len(never_marked)}")
if never_marked:
    print(f"  IDs: {sorted(never_marked)}")

---
## 9. Feature Engineering (Cleaning-Derived Columns)

In [ ]:
# ── 9.1  customers — derived columns useful for analysis ─────────────────────
# is_churned flag
customers["is_churned"] = customers["customer_id"].isin(churned_ids)

# signup year-month (period) for cohort analysis
customers["signup_ym"] = customers["signup_date"].dt.to_period("M")

# signup quarter
customers["signup_quarter"] = (
    customers["signup_date"].dt.year.astype(str) + "-Q" +
    customers["signup_date"].dt.quarter.astype(str)
)

print("[customers] derived columns added: is_churned, signup_ym, signup_quarter")
print(customers[["customer_id", "signup_date",
                  "signup_ym", "signup_quarter", "is_churned"]].head(5).to_string())

In [ ]:
# ── 9.2  revenue — effective revenue & discount rate ─────────────────────────
revenue["discount_rate"] = np.where(
    revenue["base_revenue"] > 0,
    (revenue["discount"] / revenue["base_revenue"]).round(4),
    np.nan
)

# total realized revenue including expansion
revenue["total_revenue"] = (
    revenue["subscription_revenue"] + revenue["expansion_revenue"]
).round(2)

# year-month period for time-series grouping
revenue["ym"] = revenue["month"].dt.to_period("M")

print("[revenue] derived columns: discount_rate, total_revenue, ym")
print(revenue[["customer_id", "month", "base_revenue", "discount",
               "discount_rate", "subscription_revenue",
               "expansion_revenue", "total_revenue"]].head(5).to_string())

In [ ]:
# ── 9.3  usage — engagement score & at-risk flag ─────────────────────────────
# Simple weighted engagement score (can be tuned)
usage["engagement_score"] = (
    0.4 * usage["feature_usage"] +
    0.3 * np.log1p(usage["sessions"]) +
    0.3 * np.log1p(usage["active_users"])
).round(3)

# At-risk flag: sessions < 5 AND active_users <= 1
usage["at_risk_flag"] = (
    (usage["sessions"] < 5) & (usage["active_users"] <= 1)
)

# year-month period
usage["ym"] = usage["month"].dt.to_period("M")

print("[usage] derived columns: engagement_score, at_risk_flag, ym")
print(f"  at_risk rows: {usage['at_risk_flag'].sum()} "
      f"({usage['at_risk_flag'].mean()*100:.1f}% of all usage rows)")

In [ ]:
# ── 9.4  churn — months-to-churn from signup ──────────────────────────────────
churn_with_signup = churn.merge(
    customers[["customer_id", "signup_date"]], on="customer_id", how="left"
)
churn_with_signup["tenure_at_churn_months"] = (
    (churn_with_signup["churn_date"] - churn_with_signup["signup_date"])
    .dt.days / 30.44
).round(1)

# Merge back
churn = churn.merge(
    churn_with_signup[["customer_id", "tenure_at_churn_months"]],
    on="customer_id", how="left"
)

print("[churn] derived column: tenure_at_churn_months")
print(churn[["customer_id", "churn_date", "reason",
             "tenure_at_churn_months"]].to_string())

---
## 10. Save Cleaned Datasets

In [ ]:
# Drop the IQR outlier flag columns before saving (keep data lean)
# They were for inspection only; uncomment if you want to keep them.

outlier_cols_rev = [c for c in revenue.columns if c.endswith("_outlier")]
outlier_cols_usg = [c for c in usage.columns   if c.endswith("_outlier")]

revenue_clean = revenue.drop(columns=outlier_cols_rev)
usage_clean   = usage.drop(columns=outlier_cols_usg)

# Save
customers.to_csv("customers_clean.csv",  index=False)
plans.to_csv(    "plans_clean.csv",       index=False)
revenue_clean.to_csv("revenue_clean.csv", index=False)
usage_clean.to_csv(  "usage_clean.csv",   index=False)
churn.to_csv(    "churn_clean.csv",       index=False)

print("Cleaned files saved:")
for fname, df in [("customers_clean.csv", customers),
                   ("plans_clean.csv",     plans),
                   ("revenue_clean.csv",   revenue_clean),
                   ("usage_clean.csv",     usage_clean),
                   ("churn_clean.csv",     churn)]:
    print(f"  {fname:25s}: {len(df):>6,} rows × {df.shape[1]} cols")

---
## 11. Cleaning Summary Report

In [ ]:
# ── Final summary printed as a clean table ────────────────────────────────────
summary_data = {
    "Table"        : ["customers", "plans", "revenue", "usage", "churn"],
    "Raw Rows"     : [500, 4, 9408, 9408, 36],
    "Clean Rows"   : [len(customers), len(plans), len(revenue_clean),
                      len(usage_clean), len(churn)],
    "Cols (clean)" : [customers.shape[1], plans.shape[1],
                      revenue_clean.shape[1], usage_clean.shape[1],
                      churn.shape[1]],
}

summary_df = pd.DataFrame(summary_data)
summary_df["Rows Removed"] = summary_df["Raw Rows"] - summary_df["Clean Rows"]

print("\n" + "="*65)
print("  NOVASUITE — DATA CLEANING SUMMARY")
print("="*65)
print(summary_df.to_string(index=False))

print("\n── Cleaning Steps Performed ────────────────────────────────────")
steps = [
    "1. Loaded 5 CSV files; verified shapes",
    "2. Schema inspection — dtypes, nulls, unique counts",
    "3. Hidden empty-string nulls replaced with NaN",
    "4. Full-row and key-level duplicate detection & removal",
    "5. Date columns parsed to datetime; NaT checked",
    "6. String columns stripped of whitespace",
    "7. Numeric columns coerced; coercion failures logged",
    "8. Categorical domain validation (industry, size, country, plan)",
    "9. Date range validation (signup, revenue, usage months)",
    "10. Financial rule checks (neg values, discount > base)",
    "11. subscription_revenue reconciliation vs base - discount",
    "12. Feature_usage range check [0, 10]",
    "13. Churn reason validation",
    "14. Churn date > signup date cross-check",
    "15. IQR outlier flagging on numeric columns",
    "16. Zero-usage month identification",
    "17. Referential integrity: all IDs resolve across tables",
    "18. Coverage check: every customer has revenue & usage rows",
    "19. Churn status consistency in revenue",
    "20. Derived columns: cohort periods, discount_rate,",
    "    total_revenue, engagement_score, at_risk_flag,",
    "    tenure_at_churn_months",
]
for s in steps:
    print(f"  {s}")
print("="*65)